# Phase 2 · Step 2 — AWQ (Sec. 6 item 3) — full run of the GQA-fixed model + TTFT

**Why it failed before:** the Mid-Sem attempt pinned `transformers==4.47.1`. That version is *older than Phi-4-mini
itself* — the model card requires `transformers==4.49.0` — which is what caused the `rope_scaling` error.
With the newest transformers, AutoAWQ breaks the other way (`LossKwargs` import error).
AutoAWQ's own last tested version is **`transformers==4.51.3`**, which is *also* new enough for Phi-4-mini.
So this notebook pins exactly that, in a fresh Colab session (the "isolated environment" from Table 2).

**Controls identical to Phase 1:** same 200 + 200 questions, prompts, chat template, extractors, greedy decoding,
512 new tokens, and memory measured the same way as the bnb runs (`torch.cuda.max_memory_allocated`).
Results are written to the same Drive folder as Step 1, so the tables notebook picks them up automatically.

**How to run** (T4 GPU runtime, **new session**):
1. **Runtime → Run all.** Cell 1 installs the pinned versions and then **stops on purpose** with a red "INSTALLED" message.
2. **Runtime → Restart session**, then **Runtime → Run all** again. Cell 1 now prints `OK transformers 4.51.3 ...` and everything continues.

In [ ]:
# ==== 1. Pinned environment (no automatic restart — you restart once by hand) ====
import importlib.metadata as im, os, subprocess, sys
WANT, MARK = "4.51.3", "/content/.awq_env_installed"
def ver(pkg):
    try: return im.version(pkg)
    except im.PackageNotFoundError: return None

if ver("transformers") == WANT and ver("autoawq"):
    import torch
    print("OK  transformers", ver("transformers"), "| autoawq", ver("autoawq"), "| torch", torch.__version__,
          "| GPU:", torch.cuda.get_device_name(0))
elif not os.path.exists(MARK):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"transformers=={WANT}", "autoawq==0.2.9", "huggingface_hub<1.0",
                    "datasets", "accelerate", "zstandard"], check=True)
    open(MARK, "w").write("done")
    raise SystemExit("INSTALLED.  Now: Runtime -> Restart session, then Runtime -> Run all.  "
                     "(This red message is expected.)")
else:
    print(subprocess.run([sys.executable, "-m", "pip", "show", "transformers", "autoawq"],
                         capture_output=True, text=True).stdout)
    raise SystemExit(f"Versions still wrong after install (transformers={ver('transformers')}). Send this output to Claude.")

In [ ]:
# ==== 2. Settings + Drive ====
SMOKE_TEST = False       # the GQA-fixed model already passed its 10-question check -> full 200 + 200 run
VARIANT    = "awq_w4"
MODEL_ID   = "microsoft/Phi-4-mini-instruct"
QUANT_CONFIG = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}   # standard AWQ W4 g128
CACHE_AWQ_ON_DRIVE = True    # the quantized model (~2.5 GB) is saved to Drive so quantizing happens only once

import json, time, re, gc, shutil, random
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/quant_study_phase2"
except Exception as e:
    print("!! Drive not mounted:", e, "\n!! results stay in this session — run the last cell to download them")
    DRIVE = "/content/no_drive"
RUN_DIR = f"{DRIVE}/runs_smoke" if SMOKE_TEST else f"{DRIVE}/runs"
AWQ_LOCAL = "/content/phi4mini-awq"
AWQ_DRIVE = f"{DRIVE}/awq_model_gqafix"     # NEW folder: the old awq_model was built with the buggy recipe
# before the fixed model exists, move attempt-1 (broken) answers aside — kept as evidence for the report
if not os.path.exists(AWQ_DRIVE):
    for d in (f"{DRIVE}/runs_smoke", f"{DRIVE}/runs"):
        for t in ("gsm8k", "mmlu"):
            old_f = f"{d}/{VARIANT}__{t}.jsonl"
            if os.path.exists(old_f):
                os.makedirs(f"{DRIVE}/awq_attempt1_broken", exist_ok=True)
                shutil.move(old_f, f"{DRIVE}/awq_attempt1_broken/{os.path.basename(d)}__{t}.jsonl")
                print("moved attempt-1 file aside:", old_f)
os.makedirs(RUN_DIR, exist_ok=True)
N_LIMIT = 10 if SMOKE_TEST else None
print("results ->", RUN_DIR)

In [ ]:
# ==== 3. Same questions as Phase 1 ====
from datasets import load_dataset
random.seed(42)
gsm8k = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=42).select(range(200))
def extract_gsm8k_answer(a): return a.split("####")[-1].strip().replace(",", "")
gsm8k_questions = [{"question": ex["question"], "gold": extract_gsm8k_answer(ex["answer"])} for ex in gsm8k]
mmlu = load_dataset("cais/mmlu", "all", split="test").shuffle(seed=42).select(range(200))
letters = ["A", "B", "C", "D"]
mmlu_questions = [{"question": f"{ex['question']}\n" + "\n".join(f"{letters[i]}. {c}" for i, c in enumerate(ex["choices"])),
                   "gold": letters[ex["answer"]], "subject": ex["subject"]} for ex in mmlu]
assert gsm8k_questions[0]["gold"] == "109" and mmlu_questions[0]["subject"] == "college_physics", "subset differs from Phase 1!"
print("same subsets as Phase 1")

In [ ]:
# ==== 4. Quantize with AWQ (skipped if a finished AWQ model is already on Drive) ====
import torch
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def awq_ready(p): return os.path.exists(f"{p}/config.json") and any(f.endswith(".safetensors") for f in os.listdir(p))

if not (os.path.isdir(AWQ_LOCAL) and awq_ready(AWQ_LOCAL)):
    if os.path.isdir(AWQ_DRIVE) and awq_ready(AWQ_DRIVE):
        shutil.copytree(AWQ_DRIVE, AWQ_LOCAL, dirs_exist_ok=True); print("AWQ model restored from Drive")
    else:
        from awq import AutoAWQForCausalLM
        from awq.models.phi3 import Phi3AWQForCausalLM
        # ---- FIX for Phi-4-mini (GQA: 24 query heads, 8 KV heads) ----
        # AutoAWQ's Phi-3 recipe always moves a scale from o_proj into the LAST rows of the fused qkv_proj,
        # assuming those rows are V. With GQA, V is only the last 1024 of 5120 rows, so the scale also hits
        # Q and K rows -> the "equivalent" transform is not equivalent -> garbage output (attempt 1).
        # AutoAWQ's Llama recipe skips this step when shapes differ; we do the same here.
        _orig = Phi3AWQForCausalLM.get_layers_for_scaling
        def _gqa_safe(module, input_feat, module_kwargs):
            layers = _orig(module, input_feat, module_kwargs)
            attn = module.self_attn
            if attn.qkv_proj.out_features != 3 * attn.o_proj.in_features:     # GQA -> skip qkv->o_proj scaling
                layers = [l for l in layers if not (l["prev_op"] is attn.qkv_proj)]
            return layers
        Phi3AWQForCausalLM.get_layers_for_scaling = staticmethod(_gqa_safe)
        print("AutoAWQ Phi-3 recipe patched for GQA (qkv_proj -> o_proj scaling skipped)")
        t0 = time.time()
        # trust_remote_code=False -> use the native transformers Phi-3 code, NOT the Hub's custom file
        awq_model = AutoAWQForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=False, safetensors=True,
                                                       low_cpu_mem_usage=True, use_cache=False)
        awq_model.quantize(tokenizer, quant_config=QUANT_CONFIG)    # calibration: pile-val, 128 samples (AWQ default)
        awq_model.save_quantized(AWQ_LOCAL)
        tokenizer.save_pretrained(AWQ_LOCAL)
        print(f"AWQ quantization took {(time.time()-t0)/60:.1f} min")
        del awq_model; gc.collect(); torch.cuda.empty_cache()
        if CACHE_AWQ_ON_DRIVE and DRIVE.startswith("/content/drive"):
            shutil.copytree(AWQ_LOCAL, AWQ_DRIVE, dirs_exist_ok=True); print("saved to Drive:", AWQ_DRIVE)
print(sorted(os.listdir(AWQ_LOCAL)))

In [ ]:
# ==== 5. Load the AWQ model exactly like the Phase-1 models (transformers + generate) ====
from transformers import AutoModelForCausalLM
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
t0 = time.time()
model_awq = AutoModelForCausalLM.from_pretrained(AWQ_LOCAL, device_map="cuda", torch_dtype=torch.float16)
load_time = time.time() - t0
print("quantization_config:", model_awq.config.quantization_config)
print(f"loaded in {load_time:.0f}s, allocated {torch.cuda.memory_allocated()/1024**3:.2f} GB")

def make_generate_fn(model, tok):          # identical to the Phase-1 notebook
    def generate_fn(prompt, max_new_tokens):
        inputs = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True).to(model.device)
        in_len = inputs["input_ids"].shape[1]
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        return tok.decode(out[0][in_len:], skip_special_tokens=True), out.shape[1] - in_len
    return generate_fn
gen_awq = make_generate_fn(model_awq, tokenizer)
print(repr(gen_awq("What is 12*7? Give 'Answer: <number>'.", 64)[0]))

In [ ]:
# ==== 6. Benchmark (Phase-1 scoring, resumable, one JSON line per question on Drive) ====
GSM8K_PROMPT = ("Solve this math problem step by step, then give the final numeric answer "
                "on the last line as 'Answer: <number>'.\n\nProblem: {q}\n")
MMLU_PROMPT  = ("Answer this multiple-choice question. Respond with only the letter "
                "(A, B, C, or D) on the last line as 'Answer: <letter>'.\n\nQuestion: {q}\n")
def extract_number(text):
    text = text.replace(",", ""); m = re.findall(r"Answer:\s*(-?\d+\.?\d*)", text)
    if m: return m[-1]
    nums = re.findall(r"-?\d+\.?\d*", text); return nums[-1] if nums else None
def extract_letter(text):
    m = re.findall(r"Answer:\s*([ABCD])", text.upper())
    if m: return m[-1]
    m = re.findall(r"\b([ABCD])\b", text.upper()); return m[-1] if m else None

@torch.no_grad()
def run_resumable(task, questions, limit=None):
    qs = questions[:limit] if limit else questions
    path = f"{RUN_DIR}/{VARIANT}__{task}.jsonl"
    done = {json.loads(l)["idx"]: json.loads(l) for l in open(path)} if os.path.exists(path) else {}
    todo = [i for i in range(len(qs)) if i not in done]
    print(f"[{VARIANT}|{task}] {len(done)} done, {len(todo)} to go")
    torch.cuda.reset_peak_memory_stats()
    with open(path, "a") as f:
        for n, i in enumerate(todo):
            ex = qs[i]
            prompt = (GSM8K_PROMPT if task == "gsm8k" else MMLU_PROMPT).format(q=ex["question"])
            t0 = time.time(); text, n_tok = gen_awq(prompt, 512); dt = time.time() - t0
            pred = extract_number(text) if task == "gsm8k" else extract_letter(text)
            rec = {"idx": i, "gold": ex["gold"], "pred": pred,
                   "correct": (pred is not None) and str(pred).strip() == str(ex["gold"]).strip(),
                   "n_tokens": int(n_tok), "wall_s": dt, "subject": ex.get("subject"), "output": text}
            f.write(json.dumps(rec) + "\n"); f.flush(); done[i] = rec
            if (n + 1) % 20 == 0:
                print(f"  ...{len(done)}/{len(qs)}  running acc {100*sum(r['correct'] for r in done.values())/len(done):.1f}%")
    recs = list(done.values())
    peak = torch.cuda.max_memory_allocated() / 1024**3 if todo else 0.0
    print(f"[{VARIANT}|{task}] acc={100*sum(r['correct'] for r in recs)/len(recs):.1f}%  "
          f"tok/s={sum(r['n_tokens'] for r in recs)/sum(r['wall_s'] for r in recs):.1f}  peak={peak:.2f} GB")
    return peak

peaks = [run_resumable("gsm8k", gsm8k_questions, N_LIMIT), run_resumable("mmlu", mmlu_questions, N_LIMIT)]
meta_p = f"{RUN_DIR}/{VARIANT}__meta.json"
meta = json.load(open(meta_p)) if os.path.exists(meta_p) else {}
meta.update({"peak_mem_gb": round(max(peaks + [meta.get("peak_mem_gb", 0)]), 3), "runtime": "transformers+autoawq",
             "weights": "AWQ W4 g128 (GEMM)", "kv": "f16", "transformers": WANT, "load_time_s": round(load_time, 1)})
json.dump(meta, open(meta_p, "w"), indent=1); print(meta)

# ---- safety check: is the AWQ model producing real answers? (attempt 1 produced only repeated words) ----
recs = [json.loads(l) for t in ("gsm8k", "mmlu") for l in open(f"{RUN_DIR}/{VARIANT}__{t}.jsonl")]
n_ok = sum(r["correct"] for r in recs); n_max = sum(r["n_tokens"] >= 512 for r in recs)
print("\nSample output:", repr(recs[0]["output"][:300]))
if n_ok == 0 or n_max > 0.5 * len(recs):
    print(f"!! STOP: {n_ok}/{len(recs)} correct, {n_max}/{len(recs)} answers hit the 512-token limit. "
          "The AWQ model looks broken — do NOT run the full set; send this output to Claude.")
elif SMOKE_TEST:
    print(f"OK: {n_ok}/{len(recs)} correct, outputs look normal. Set SMOKE_TEST = False in cell 2 and Run all again.")
else:
    print("Full AWQ run finished.")

In [ ]:
# ==== 6b. Time to first token (TTFT) for AWQ: 30 GSM8K + 30 MMLU prompts, 1 generated token each ====
TTFT_PATH = f"{DRIVE}/runs/ttft_awq.json"
if not os.path.exists(TTFT_PATH):
    import statistics as stx
    def ttft_hf(prompt):
        inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                               return_tensors="pt", return_dict=True).to(model_awq.device)
        torch.cuda.synchronize(); t0 = time.time()
        with torch.no_grad():
            model_awq.generate(**inputs, max_new_tokens=1, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        torch.cuda.synchronize()
        return (time.time() - t0) * 1000, inputs["input_ids"].shape[1]
    ttft_hf(GSM8K_PROMPT.format(q=gsm8k_questions[0]["question"]))          # warm-up
    out = {}
    for task, qs, P in [("gsm8k", gsm8k_questions, GSM8K_PROMPT), ("mmlu", mmlu_questions, MMLU_PROMPT)]:
        r = [ttft_hf(P.format(q=q["question"])) for q in qs[:30]]
        out[task] = {"median_ttft_ms": round(stx.median(x for x, _ in r), 1), "median_prompt_tokens": stx.median(n for _, n in r)}
    json.dump({"awq_w4": out}, open(TTFT_PATH, "w"), indent=1)
print(json.load(open(TTFT_PATH)))

In [ ]:
# ==== 7. Download results (needed only if Drive is not mounted) ====
if DRIVE.startswith("/content/drive"):
    print("Drive is mounted — results are already saved in", RUN_DIR, "(no download needed)")
else:
    shutil.make_archive("/content/awq_results", "zip", RUN_DIR)
    from google.colab import files; files.download("/content/awq_results.zip")